# Clustered Randomization with `clustered-experiments`

This is part of the [CausalPython](https://causalpython.io) series on causality.

<a href="https://causalpython.io"><img src="img/CausalPython.io__flat.png" width=150 align="left"></a>

<br>

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
from cluster_experiments import AnalysisPlan

## 2. Synthetic DGP

In [3]:
# Define parameters
n_stores = 500  # Number of stores (clusters)
transactions_per_store = 100  # Average transactions per store

# Step 1: Randomly assign stores to treatment
stores = pd.DataFrame({
    'store_id': range(n_stores),
    'variant': np.random.choice(['control', 'treatment'], n_stores),
})

# Step 2: Generate transaction-level data
transactions = []
for _, store in stores.iterrows():
    n_transactions = np.random.poisson(transactions_per_store)
    
    # Base purchase amount
    base_amount = 50
    
    # Treatment effect: +$5 average purchase
    treatment_effect = 5 if store['variant'] == 'treatment' else 0
    
    # Store-level random effect (intra-cluster correlation)
    store_effect = np.random.normal(0, 10)
    
    # Generate transactions
    store_transactions = pd.DataFrame({
        'store_id': store['store_id'],
        'variant': store['variant'],
        'purchase_amount': np.random.normal(
            base_amount + treatment_effect + store_effect, 
            20, 
            n_transactions
        ).clip(min=0)  # No negative purchases
    })
    
    transactions.append(store_transactions)

data = pd.concat(transactions, ignore_index=True)

print(f"Total transactions: {len(data):,}")
print(f"Stores in control: {(stores['variant'] == 'control').sum()}")
print(f"Stores in treatment: {(stores['variant'] == 'treatment').sum()}")
print(f"\nFirst few rows:")
data.head()


Total transactions: 50,071
Stores in control: 244
Stores in treatment: 256

First few rows:


,store_id,variant,purchase_amount
0,0,control,68.589627
1,0,control,58.216537
2,0,control,88.492218
3,0,control,73.454058
4,0,control,41.721246


## 3. Clustered Analysis

In [4]:
# Correct analysis with clustered standard errors
clustered_plan = AnalysisPlan.from_metrics_dict({
    'metrics': [
        {
            'alias': 'purchase_amount',
            'name': 'purchase_amount',
            'metric_type': 'simple'
        },
    ],
    'variants': [
        {'name': 'control', 'is_control': True},
        {'name': 'treatment', 'is_control': False},
    ],
    'variant_col': 'variant',
    'analysis_type': 'clustered_ols',  # Clustered OLS (CORRECT!)
    'analysis_config': {
        'cluster_cols': ['store_id']  # Specify the clustering variable
    }
})

clustered_results = clustered_plan.analyze(data).to_dataframe()
print("=== Correct Analysis (With Clustering) ===")
print(f"Treatment Effect: ${clustered_results.iloc[0]['ate']:.2f}")
print(f"Standard Error: ${clustered_results.iloc[0]['std_error']:.2f}")
print(f"P-value: {clustered_results.iloc[0]['p_value']:.4f}")
print(f"95% CI: [${clustered_results.iloc[0]['ate_ci_lower']:.2f}, ${clustered_results.iloc[0]['ate_ci_upper']:.2f}]")


=== Correct Analysis (With Clustering) ===
Treatment Effect: $3.60
Standard Error: $0.87
P-value: 0.0000
95% CI: [$1.89, $5.31]
